# Lesson 8 — Dates & time series
**Goal:** turn timestamps into insight: monthly sales trend, best weekday, busiest hour for mobile money, and a rolling average.

In [1]:
import pandas as pd
orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
orders['revenue'] = orders['quantity'] * orders['unit_price'] * (1 - orders['discount_pct']/100)
mm = pd.read_csv('data/mobile_money.csv', parse_dates=['timestamp'])

## The `.dt` accessor

In [2]:
orders['month'] = orders['order_date'].dt.month_name()
orders['weekday'] = orders['order_date'].dt.day_name()
orders[['order_date', 'month', 'weekday']].head(3)

,order_date,month,weekday
0,2025-01-01,January,Wednesday
1,2025-01-01,January,Wednesday
2,2025-01-01,January,Wednesday


## `resample` — monthly revenue
Set the date as the index, then resample by month-end (`'ME'`).

In [3]:
monthly = orders.set_index('order_date')['revenue'].resample('ME').sum()
(monthly / 1e6).round(1).rename('revenue_TSh_millions')

order_date
2025-01-31    19.0
2025-02-28    16.8
2025-03-31    18.2
2025-04-30    18.5
2025-05-31    26.8
2025-06-30    27.1
2025-07-31    27.5
2025-08-31    27.1
2025-09-30    30.1
2025-10-31    30.3
2025-11-30    32.9
2025-12-31    45.4
Freq: ME, Name: revenue_TSh_millions, dtype: float64

In [4]:
growth = monthly.pct_change().mul(100).round(1)
print('Best month:', monthly.idxmax().strftime('%B'), '| Dec vs Nov growth:', growth.iloc[-1], '%')

Best month: December | Dec vs Nov growth: 38.0 %


## Which weekday sells most?

In [5]:
order_of_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
orders.groupby('weekday')['revenue'].sum().reindex(order_of_days).div(1e6).round(1)

weekday
Monday       51.4
Tuesday      51.8
Wednesday    47.9
Thursday     42.0
Friday       40.2
Saturday     49.7
Sunday       36.7
Name: revenue, dtype: float64

## Rolling averages smooth out daily noise

In [6]:
daily = orders.set_index('order_date')['revenue'].resample('D').sum()
smooth = daily.rolling(7).mean()
pd.DataFrame({'daily': daily, '7-day avg': smooth}).tail(7).round(0)

,daily,7-day avg
order_date,,
2025-12-25,615800.0,1399236.0
2025-12-26,1790200.0,1554057.0
2025-12-27,997750.0,1253179.0
2025-12-28,1201200.0,1236300.0
2025-12-29,1272600.0,1248507.0
2025-12-30,1722950.0,1160964.0
2025-12-31,1702250.0,1328964.0


## Mobile money: busiest hour of the day

In [7]:
mm['hour'] = mm['timestamp'].dt.hour
mm.groupby('hour')['txn_id'].count().sort_values(ascending=False).head(5)

hour
9     132
12    121
19    117
8     114
17    108
Name: txn_id, dtype: int64

In [8]:
# Filter by date range with .loc on a DatetimeIndex
q4 = orders.set_index('order_date').loc['2025-10-01':'2025-12-31']
print('Q4 revenue: TSh', f"{q4['revenue'].sum():,.0f}")

Q4 revenue: TSh 108,616,400


## Key takeaways
- Always `parse_dates`; then use `.dt.year/.month/.day_name()/.hour`.
- `resample('D'|'W'|'ME'|'QE')` for time buckets; `pct_change` for growth; `rolling(n)` for smoothing.

**Exercise:** what share of yearly revenue came in Q4?